# Regularization & Training Dynamics

Training a deep neural network is not just about defining the architecture — **how** you train it matters enormously.  
A model with 10 million parameters trained carelessly will underperform a 1 million parameter model trained well.

This notebook covers the practical science of making networks train reliably and generalize to unseen data.

---
**Topics covered**
1. Overfitting and the bias-variance tradeoff
2. L1 and L2 weight regularization (Ridge & Lasso)
3. Dropout — random neuron deactivation
4. Batch Normalization — normalizing activations per mini-batch
5. Data augmentation — creating artificial training variety
6. Learning rate scheduling — warm-up, step decay, cosine annealing
7. Gradient clipping — preventing exploding gradients
8. Early stopping and model checkpointing
9. Full experiment: comparing regularization strategies on CIFAR-10
10. Practical training checklist

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    TORCH = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'PyTorch {torch.__version__} | device: {device}')
except ImportError:
    TORCH = False
    print('PyTorch not found — NumPy demos will still run.')

## 1. Overfitting and the Bias-Variance Tradeoff

A model that **memorizes** the training data rather than learning generalizable patterns will fail on new data.  
This is **overfitting**.

```
  Underfitting (high bias):         Good fit:              Overfitting (high variance):

  Model is too simple.              Captures the true       Model is too complex.
  Misses real patterns.             underlying pattern.     Memorizes noise.

  Training loss: high               Training loss: low      Training loss: very low
  Val loss:      high               Val loss:      low      Val loss:      high
```

The **bias-variance decomposition**:

$$\text{Expected Error} = \underbrace{\text{Bias}^2}_{\text{underfitting}} + \underbrace{\text{Variance}}_{\text{overfitting}} + \text{Irreducible Noise}$$

Regularization adds **constraints** that reduce variance (overfitting) at the cost of slightly increased bias.

In [ ]:
# Generate a noisy sine wave dataset
np.random.seed(7)
n = 30
X_tr = np.linspace(0, 2*np.pi, n)
y_tr = np.sin(X_tr) + np.random.randn(n) * 0.3
X_te = np.linspace(0, 2*np.pi, 200)
y_te = np.sin(X_te)

def poly_fit(X_tr, y_tr, X_te, degree):
    coeffs = np.polyfit(X_tr, y_tr, degree)
    return np.polyval(coeffs, X_te)

degrees = [1, 4, 20]
titles  = ['Underfitting\n(degree 1 — high bias)',
           'Good Fit\n(degree 4)',
           'Overfitting\n(degree 20 — high variance)']
colors  = ['#F44336', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, deg, title, col in zip(axes, degrees, titles, colors):
    y_pred = poly_fit(X_tr, y_tr, X_te, deg)
    ax.scatter(X_tr, y_tr, color='gray', s=30, zorder=3, label='Training data')
    ax.plot(X_te, y_te,   'k--', lw=1.5, label='True function', alpha=0.5)
    ax.plot(X_te, y_pred, color=col, lw=2.5, label=f'Model (deg={deg})')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff: Underfitting vs Good Fit vs Overfitting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Training vs validation error curve
model_complexity = np.arange(1, 21)
train_err = 1.5 * np.exp(-0.3 * model_complexity) + 0.05
val_err   = 0.3 * np.exp(-0.3 * model_complexity) + 0.1 + 0.008 * model_complexity

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(model_complexity, train_err, 'b-o', lw=2, markersize=5, label='Training Error')
ax.plot(model_complexity, val_err,   'r-s', lw=2, markersize=5, label='Validation Error')
opt = np.argmin(val_err)
ax.axvline(model_complexity[opt], color='green', linestyle='--', lw=2, label=f'Optimal complexity = {model_complexity[opt]}')
ax.fill_betweenx([0, 2], 0, model_complexity[opt]*0.5, alpha=0.1, color='blue', label='High bias region')
ax.fill_betweenx([0, 2], model_complexity[opt]*1.5, 21, alpha=0.1, color='red', label='High variance region')
ax.set_xlabel('Model Complexity', fontsize=11)
ax.set_ylabel('Error', fontsize=11)
ax.set_title('Bias-Variance Tradeoff Curve', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(1, 20); ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()

## 2. L1 and L2 Weight Regularization

Add a **penalty term** to the loss function to discourage large weights:

| Method | Loss | Effect on weights | Use case |
|---|---|---|---|
| **L2 (Ridge / weight decay)** | $L + \lambda \sum w_i^2$ | Shrinks all weights toward 0 | Default; prevents any single feature from dominating |
| **L1 (Lasso)** | $L + \lambda \sum \|w_i\|$ | Drives many weights to exactly 0 (sparse) | Feature selection; many irrelevant inputs |
| **Elastic Net** | $L + \lambda_1\|w\|_1 + \lambda_2\|w\|_2^2$ | Combines both | When both sparsity and grouping are desired |

### L2 Gradient Update

$$w \leftarrow w - \eta \frac{\partial L}{\partial w} - \eta \lambda w = w(1 - \eta\lambda) - \eta \frac{\partial L}{\partial w}$$

The factor $(1 - \eta\lambda)$ **decays** the weight at each step — hence the name **weight decay**.

In [ ]:
# Visualise L1 vs L2 penalty surfaces and their effect on the optimal solution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

w = np.linspace(-3, 3, 300)

# ── Penalty functions ────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(w, w**2,    color='#2196F3', lw=2.5, label='L2: w²')
ax.plot(w, np.abs(w), color='#F44336', lw=2.5, label='L1: |w|')
ax.set_title('Penalty Functions', fontweight='bold')
ax.set_xlabel('Weight value'); ax.set_ylabel('Penalty')
ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(-3,3); ax.set_ylim(0,5)

# ── Gradient of penalty ───────────────────────────────────────────────────────
ax = axes[1]
ax.plot(w, 2*w,               color='#2196F3', lw=2.5, label='L2 gradient: 2w')
ax.plot(w, np.sign(w),        color='#F44336', lw=2.5, label='L1 gradient: sign(w)')
ax.axhline(0, color='black', lw=0.8)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Gradient of Penalty\n(added to weight update)', fontweight='bold')
ax.set_xlabel('Weight value'); ax.set_ylabel('Gradient')
ax.legend(); ax.grid(alpha=0.3)

# ── Contour visualization: L1 vs L2 constraint regions ───────────────────────
ax = axes[2]
theta = np.linspace(0, 2*np.pi, 300)
# L2 constraint: w1² + w2² ≤ r²  (circle)
r = 1.5
ax.plot(r*np.cos(theta), r*np.sin(theta), 'b-', lw=2.5, label='L2 constraint (sphere)')
# L1 constraint: |w1| + |w2| ≤ r  (diamond)
diamond_x = r * np.array([1, 0, -1, 0, 1])
diamond_y = r * np.array([0, 1,  0,-1, 0])
ax.plot(diamond_x, diamond_y, 'r-', lw=2.5, label='L1 constraint (diamond)')
# Loss contours (ellipses centered off-axis)
cx, cy = 2.5, 1.5
for radius in [0.5, 1.0, 1.5, 2.0, 2.5]:
    ax.plot(cx + radius * 0.8 * np.cos(theta), cy + radius * np.sin(theta),
            'gray', lw=0.8, alpha=0.5)
# Intersection points
ax.scatter([0], [r], color='red',  s=150, zorder=5, label='L1 optimum (sparse!)')
ax.scatter([r*0.7], [r*0.55], color='blue', s=150, zorder=5, label='L2 optimum')
ax.set_xlim(-3,3); ax.set_ylim(-3,3)
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Why L1 Produces Sparse Weights:\nConstraint Region Geometry', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_aspect('equal')

plt.suptitle('L1 vs L2 Regularization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
if TORCH:
    # Demonstrate L1 and L2 effect on a neural network
    # Dataset: moons (nonlinear binary classification)
    X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
    X = StandardScaler().fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    X_tr_t = torch.FloatTensor(X_tr)
    y_tr_t = torch.LongTensor(y_tr)
    X_te_t = torch.FloatTensor(X_te)
    y_te_t = torch.LongTensor(y_te)

    class MLP(nn.Module):
        def __init__(self, hidden=64):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, 2)
            )
        def forward(self, x): return self.net(x)

    def train_mlp(l2=0.0, l1=0.0, epochs=200, hidden=64):
        model = MLP(hidden)
        opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=l2)
        train_losses, val_losses = [], []
        for _ in range(epochs):
            model.train()
            logits = model(X_tr_t)
            loss = F.cross_entropy(logits, y_tr_t)
            if l1 > 0:
                l1_penalty = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1 * l1_penalty
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                tl = F.cross_entropy(model(X_tr_t), y_tr_t).item()
                vl = F.cross_entropy(model(X_te_t), y_te_t).item()
            train_losses.append(tl); val_losses.append(vl)
        model.eval()
        with torch.no_grad():
            acc = (model(X_te_t).argmax(1) == y_te_t).float().mean().item()
        return model, train_losses, val_losses, acc

    configs = [
        ('No Reg',  0.0,  0.0,  '#9E9E9E'),
        ('L2=1e-3', 1e-3, 0.0,  '#2196F3'),
        ('L2=1e-1', 1e-1, 0.0,  '#9C27B0'),
        ('L1=1e-4', 0.0,  1e-4, '#FF9800'),
    ]

    results = {}
    for name, l2, l1, col in configs:
        model, trl, vll, acc = train_mlp(l2=l2, l1=l1)
        results[name] = {'model': model, 'train': trl, 'val': vll, 'acc': acc, 'color': col}
        print(f'{name:<12}  val_acc={acc*100:.1f}%  '
              f'final_gap={vll[-1]-trl[-1]:.3f}  (smaller gap = less overfitting)')

    # Plot loss curves
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, res in results.items():
        axes[0].plot(res['train'], '--',  color=res['color'], alpha=0.6, lw=1.2)
        axes[0].plot(res['val'],   '-',   color=res['color'], lw=2, label=name)
    axes[0].set_title('Loss Curves (solid=val, dashed=train)', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Decision boundaries
    xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

    for i, (name, res) in enumerate(results.items()):
        res['model'].eval()
        with torch.no_grad():
            probs = res['model'](grid).softmax(1)[:,1].numpy().reshape(xx.shape)
        axes[1].contourf(xx, yy, probs, levels=50, cmap='RdBu', alpha=0.3)

    # Use No Reg vs L2=1e-3 for clean boundary comparison
    fig2, axes2 = plt.subplots(1, 4, figsize=(16, 3))
    for ax, (name, res) in zip(axes2, results.items()):
        with torch.no_grad():
            probs = res['model'](grid).softmax(1)[:,1].numpy().reshape(xx.shape)
        ax.contourf(xx, yy, probs, levels=20, cmap='RdBu', alpha=0.8)
        ax.scatter(X_te[y_te==0, 0], X_te[y_te==0, 1], c='blue', s=20, edgecolors='white', lw=0.5)
        ax.scatter(X_te[y_te==1, 0], X_te[y_te==1, 1], c='red',  s=20, edgecolors='white', lw=0.5)
        ax.set_title(f'{name}\nacc={res["acc"]*100:.1f}%', fontweight='bold', fontsize=9)
        ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.axis('off')
    plt.suptitle('Decision Boundaries — Effect of Regularization Strength', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 3. Dropout — Random Neuron Deactivation

**Srivastava et al., 2014** — one of the most effective and widely-used regularization techniques.

### How it works

During **training**: randomly set each neuron's activation to zero with probability `p` (typical: `p=0.5` for Fully Connected, `p=0.1–0.3` for Conv).  
During **inference**: use all neurons, but scale outputs by `(1-p)` to match expected values.

```
  Training (p=0.5):          Inference:
  ┌──────────────┐           ┌──────────────┐
  │ x₁ × mask₁   │           │ x₁ × (1-p)   │
  │ x₂ × 0 (!)   │   →   →   │ x₂ × (1-p)   │
  │ x₃ × mask₃   │           │ x₃ × (1-p)   │
  │ x₄ × 0 (!)   │           │ x₄ × (1-p)   │
  └──────────────┘           └──────────────┘
  Random mask each batch     Fixed scaling factor
```

**Why it helps:**
- Forces each neuron to be useful **independently** — no co-adaptation
- Equivalent to training an **ensemble** of 2^N different sparse networks
- Acts as **implicit L2 regularization** under certain conditions

Modern PyTorch uses **inverted dropout**: scale training activations by `1/(1-p)` so inference needs no rescaling.

In [ ]:
# Visualise dropout: show which neurons are dropped in each batch
np.random.seed(42)
n_neurons = 10
n_batches = 5
p = 0.5

fig, axes = plt.subplots(1, n_batches + 1, figsize=(14, 3))

# Full network
ax = axes[0]
ax.imshow(np.ones((n_neurons, 1)), cmap='Greens', vmin=0, vmax=1, aspect='auto')
ax.set_title('Full\nNetwork', fontweight='bold', fontsize=9)
ax.set_yticks(range(n_neurons))
ax.set_yticklabels([f'n{i}' for i in range(n_neurons)], fontsize=7)
ax.set_xticks([])

for b in range(n_batches):
    mask = np.random.binomial(1, 1-p, n_neurons)
    ax = axes[b+1]
    colors = np.array([[0.2, 0.7, 0.2]] * n_neurons)  # green = active
    colors[mask == 0] = [0.9, 0.3, 0.3]               # red = dropped
    ax.imshow(mask.reshape(-1, 1), cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    active = mask.sum()
    ax.set_title(f'Batch {b+1}\n({active}/{n_neurons} active)', fontweight='bold', fontsize=9)
    ax.set_yticks([]); ax.set_xticks([])

plt.suptitle(f'Dropout (p={p}): Different Random Masks Each Batch = Ensemble of Sparse Networks',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

if TORCH:
    # Demonstrate model.train() vs model.eval() dropout behavior
    torch.manual_seed(42)
    dropout = nn.Dropout(p=0.5)
    x = torch.ones(1, 10)

    print('Inverted dropout (PyTorch): scale by 1/(1-p) during training')
    print(f'\nInput: {x}')

    dropout.train()
    out_train = dropout(x)
    print(f'Train output: {out_train}  (active neurons scaled by 2.0 = 1/0.5)')

    dropout.eval()
    out_eval = dropout(x)
    print(f'Eval output:  {out_eval}  (no dropout, no scaling — identical to input)')

## 4. Batch Normalization — Normalizing Activations

**Ioffe & Szegedy, 2015** — arguably the most impactful training technique since ReLU.

### The Problem: Internal Covariate Shift

As the network trains, the distribution of each layer's inputs changes continuously — earlier layers' weight updates shift the statistics of later layers' inputs.  
This **covariate shift** slows training and requires careful initialization and learning rate tuning.

### BatchNorm Solution

After each layer (before activation), normalize the activations across the mini-batch:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

Then **scale and shift** with learnable parameters γ (scale) and β (shift):

$$y_i = \gamma \hat{x}_i + \beta$$

| Step | What it does |
|---|---|
| Normalize | Zero mean, unit variance within the batch |
| Scale (γ) | Learned: how spread-out the activations should be |
| Shift (β) | Learned: where the center of activations should be |

**Benefits:** faster training, higher learning rates, acts as regularizer (reduces need for Dropout), less sensitive to initialization.

In [ ]:
# Implement BatchNorm from scratch to understand the math
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    """
    x: (N, D) — N samples, D features
    Returns normalized output and intermediate values for backprop.
    """
    mu    = x.mean(axis=0)          # mean per feature  (D,)
    var   = x.var(axis=0)           # variance per feature
    x_hat = (x - mu) / np.sqrt(var + eps)   # normalize
    y     = gamma * x_hat + beta            # scale + shift
    return y, mu, var, x_hat

# Demo: a batch of 8 samples, 3 features with very different scales
np.random.seed(42)
batch = np.random.randn(8, 3) * np.array([10, 0.1, 100])  # very different scales!
gamma = np.ones(3)   # initially identity
beta  = np.zeros(3)

normed, mu, var, _ = batchnorm_forward(batch, gamma, beta)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].boxplot([batch[:, i] for i in range(3)],
                labels=['Feature 1\n(scale~10)', 'Feature 2\n(scale~0.1)', 'Feature 3\n(scale~100)'])
axes[0].set_title('Before BatchNorm\n(very different scales)', fontweight='bold')
axes[0].set_ylabel('Activation value')

axes[1].boxplot([normed[:, i] for i in range(3)],
                labels=['Feature 1', 'Feature 2', 'Feature 3'])
axes[1].set_title('After BatchNorm\n(normalized to ~N(0,1))', fontweight='bold')
axes[1].set_ylabel('Activation value')

# Training speed comparison: with vs without BN
epochs = np.arange(50)
loss_no_bn = 2.5 * np.exp(-0.03 * epochs) + 0.3 + np.random.randn(50) * 0.05
loss_bn    = 2.5 * np.exp(-0.12 * epochs) + 0.1 + np.random.randn(50) * 0.03
axes[2].plot(epochs, loss_no_bn, color='#F44336', lw=2, label='Without BatchNorm')
axes[2].plot(epochs, loss_bn,    color='#4CAF50', lw=2, label='With BatchNorm')
axes[2].set_title('Training Speed: With vs Without BatchNorm', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Batch Normalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Before BN — Feature means:    {batch.mean(0).round(2)}')
print(f'Before BN — Feature stds:     {batch.std(0).round(2)}')
print(f'\nAfter BN  — Feature means:    {normed.mean(0).round(4)}')
print(f'After BN  — Feature stds:     {normed.std(0).round(4)}')

In [ ]:
if TORCH:
    # Train with and without BatchNorm side by side
    from torchvision.datasets import MNIST

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_ds = MNIST('./data', train=True,  download=True, transform=transform)
    test_ds  = MNIST('./data', train=False, download=True, transform=transform)
    train_ld = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
    test_ld  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)

    class CNN(nn.Module):
        def __init__(self, use_bn=True, use_dropout=True):
            super().__init__()
            def conv_block(in_ch, out_ch):
                layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]
                if use_bn: layers.append(nn.BatchNorm2d(out_ch))
                layers.append(nn.ReLU(inplace=True))
                return layers

            self.features = nn.Sequential(
                *conv_block(1, 32),
                nn.MaxPool2d(2),
                *conv_block(32, 64),
                nn.MaxPool2d(2),
            )
            fc_layers = [nn.Flatten(), nn.Linear(64*7*7, 256), nn.ReLU()]
            if use_dropout: fc_layers.append(nn.Dropout(0.5))
            fc_layers.append(nn.Linear(256, 10))
            self.classifier = nn.Sequential(*fc_layers)

        def forward(self, x):
            return self.classifier(self.features(x))

    def run_experiment(use_bn, use_dropout, epochs=5, name=''):
        model = CNN(use_bn, use_dropout).to(device)
        opt = optim.Adam(model.parameters(), lr=1e-3)
        val_accs = []
        for _ in range(epochs):
            model.train()
            for X, y in train_ld:
                X, y = X.to(device), y.to(device)
                loss = F.cross_entropy(model(X), y)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            correct = 0
            with torch.no_grad():
                for X, y in test_ld:
                    X, y = X.to(device), y.to(device)
                    correct += (model(X).argmax(1) == y).sum().item()
            val_accs.append(correct / len(test_ds) * 100)
        print(f'{name:<30}  final acc={val_accs[-1]:.2f}%')
        return val_accs

    experiments = [
        (False, False, 'No BN, No Dropout'),
        (True,  False, 'BatchNorm only'),
        (False, True,  'Dropout only'),
        (True,  True,  'BN + Dropout (recommended)'),
    ]

    all_curves = {}
    print('Comparing regularization on MNIST:')
    for bn, do, name in experiments:
        all_curves[name] = run_experiment(bn, do, epochs=5, name=name)

    epochs = range(1, 6)
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#9E9E9E', '#2196F3', '#FF9800', '#4CAF50']
    for (name, accs), col in zip(all_curves.items(), colors):
        ax.plot(epochs, accs, 'o-', color=col, lw=2, label=name)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Effect of BatchNorm and Dropout on MNIST Accuracy', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Data Augmentation — Creating Training Variety

The best regularizer is **more data**. When you can't collect more, **augment** the existing data.

Data augmentation applies random transforms at training time that preserve the label:

| Transform | What changes | Invariance learned |
|---|---|---|
| Horizontal flip | Left-right mirror | Bilateral symmetry |
| Random crop | Shift + zoom | Translation |
| Random rotation | Angle | Rotation |
| Color jitter | Brightness/contrast/saturation | Lighting conditions |
| Cutout / RandomErasing | Random rectangles zeroed out | Occlusion robustness |
| MixUp | Blend two images + blend labels | Smoother decision boundaries |
| CutMix | Paste region from one image onto another | Regional occlusion |

In [ ]:
if TORCH:
    from torchvision.transforms import functional as TF
    import torchvision.transforms as T

    # Load a CIFAR-10 image
    cifar = datasets.CIFAR10('./data', train=True, download=True,
                              transform=transforms.ToTensor())
    img, label = cifar[6]   # A frog image
    classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

    # Define augmentations
    augs = [
        ('Original', lambda x: x),
        ('H-Flip', T.RandomHorizontalFlip(p=1.0)),
        ('RandomCrop', T.RandomCrop(32, padding=4)),
        ('Rotation±30°', T.RandomRotation(30)),
        ('ColorJitter', T.ColorJitter(0.4, 0.4, 0.4, 0.1)),
        ('GaussianBlur', T.GaussianBlur(3, sigma=(0.1, 2.0))),
        ('RandomErasing', T.Compose([
            T.RandomErasing(p=1.0, scale=(0.1, 0.3))
        ])),
        ('Grayscale', T.RandomGrayscale(p=1.0)),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, (name, aug) in zip(axes.flat, augs):
        torch.manual_seed(0)
        aug_img = aug(img)
        if aug_img.shape[0] == 1:   # grayscale → RGB for display
            aug_img = aug_img.repeat(3, 1, 1)
        ax.imshow(aug_img.permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title(name, fontweight='bold', fontsize=10)
        ax.axis('off')

    plt.suptitle(f'Data Augmentation — Label stays "{classes[label]}" for all transforms',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # MixUp demonstration
    img2, label2 = cifar[12]  # different class
    alpha = 0.4
    lam = np.random.beta(alpha, alpha)
    mixed = lam * img + (1 - lam) * img2

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(img.permute(1,2,0).numpy())
    axes[0].set_title(f'Image A\n(label: {classes[label]})', fontweight='bold')
    axes[1].imshow(img2.permute(1,2,0).numpy())
    axes[1].set_title(f'Image B\n(label: {classes[label2]})', fontweight='bold')
    axes[2].imshow(mixed.permute(1,2,0).clamp(0,1).numpy())
    axes[2].set_title(f'MixUp (λ={lam:.2f})\nSoft label: {lam:.2f}×{classes[label]} + {1-lam:.2f}×{classes[label2]}',
                      fontweight='bold', fontsize=8)
    for ax in axes: ax.axis('off')
    plt.suptitle('MixUp Augmentation — Trains with convex combinations of examples',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Learning Rate Scheduling

The learning rate is the most important hyperparameter.  
A **fixed LR** is rarely optimal: too high → unstable, too low → slow convergence.

### Common Schedules

| Schedule | Formula | When to use |
|---|---|---|
| **Step decay** | LR × γ every N epochs | Simple, predictable |
| **Exponential decay** | LR × γ^epoch | Smooth, continuous |
| **Cosine annealing** | LR × ½(1 + cos(πt/T)) | Most popular in practice |
| **Warmup + decay** | Linear ramp then cosine | Transformers, large LR training |
| **One-cycle** | Increase then decrease, both LR and momentum | Fast convergence (Leslie Smith) |
| **ReduceLROnPlateau** | Reduce when val loss stops improving | Automatic, model-agnostic |

In [ ]:
if TORCH:
    EPOCHS = 50
    base_lr = 0.1

    def get_lr_schedule(name, epochs=EPOCHS):
        """Return list of LRs for each epoch."""
        model = nn.Linear(1,1)  # dummy model
        lrs = []

        if name == 'Constant':
            return [base_lr] * epochs

        elif name == 'Step Decay':
            opt = optim.SGD(model.parameters(), lr=base_lr)
            sch = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
        elif name == 'Exponential':
            opt = optim.SGD(model.parameters(), lr=base_lr)
            sch = optim.lr_scheduler.ExponentialLR(opt, gamma=0.93)
        elif name == 'Cosine Annealing':
            opt = optim.SGD(model.parameters(), lr=base_lr)
            sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-4)
        elif name == 'Warmup + Cosine':
            opt = optim.SGD(model.parameters(), lr=1e-6)
            warmup = 5
            def lr_lambda(epoch):
                if epoch < warmup:
                    return (epoch + 1) / warmup * base_lr / 1e-6
                t = epoch - warmup
                T = epochs - warmup
                return 0.5 * (1 + np.cos(np.pi * t / T)) * base_lr / 1e-6
            sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        elif name == 'One-Cycle':
            opt = optim.SGD(model.parameters(), lr=base_lr/25)
            sch = optim.lr_scheduler.OneCycleLR(opt, max_lr=base_lr, total_steps=epochs)

        for _ in range(epochs):
            lrs.append(opt.param_groups[0]['lr'])
            sch.step()
        return lrs

    schedules = ['Constant', 'Step Decay', 'Exponential', 'Cosine Annealing', 'Warmup + Cosine', 'One-Cycle']
    colors     = ['#9E9E9E', '#F44336', '#FF9800', '#2196F3', '#4CAF50', '#9C27B0']

    fig, ax = plt.subplots(figsize=(13, 5))
    for name, col in zip(schedules, colors):
        lrs = get_lr_schedule(name)
        ax.plot(range(EPOCHS), lrs, lw=2.5, color=col, label=name)

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedules', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Practical recommendations:')
    print('  • Cosine Annealing — default for most CNN training')
    print('  • Warmup + Cosine  — default for large models / large batch training')
    print('  • One-Cycle        — fastest convergence (Super-Convergence)')
    print('  • ReduceLROnPlateau — good when unsure of epoch count')

## 7. Gradient Clipping — Preventing Exploding Gradients

In deep or recurrent networks, gradients can grow exponentially large during backpropagation — **gradient explosion**.  
The loss becomes `NaN` and training collapses.

### Solution: Clip Gradients Before the Update

**Norm clipping** (most common): if gradient vector L2-norm > threshold, scale all gradients proportionally:

$$g \leftarrow g \cdot \frac{\text{max\_norm}}{\|g\|_2} \quad \text{if } \|g\|_2 > \text{max\_norm}$$

**Value clipping**: clip each gradient component independently to `[-clip, +clip]`.

Norm clipping preserves gradient direction; value clipping does not.

In [ ]:
if TORCH:
    torch.manual_seed(42)

    # Simulate gradient norms over training
    model = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 1))
    opt = optim.SGD(model.parameters(), lr=0.1)

    norms_unclipped, norms_clipped = [], []
    max_norm = 1.0

    for step in range(100):
        x = torch.randn(32, 10)
        y = torch.randn(32, 1)
        loss = F.mse_loss(model(x), y)
        opt.zero_grad()
        loss.backward()

        # Compute total gradient norm before clipping
        total_norm = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)**0.5
        norms_unclipped.append(total_norm.item())

        # Clip
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        clipped_norm = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)**0.5
        norms_clipped.append(clipped_norm.item())

        opt.step()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(norms_unclipped, color='#F44336', lw=2, label='Unclipped')
    axes[0].plot(norms_clipped,   color='#4CAF50', lw=2, label='Clipped (max=1.0)')
    axes[0].axhline(max_norm, color='gray', linestyle='--', lw=1.5, label=f'max_norm={max_norm}')
    axes[0].set_title('Gradient Norm: Clipped vs Unclipped', fontweight='bold')
    axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Gradient L2 norm')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Value clipping vs norm clipping on 2D gradient
    g = np.array([3.0, 4.0])  # gradient with norm=5
    clip_val = 2.0
    g_val_clip  = np.clip(g, -clip_val, clip_val)
    g_norm_clip = g * (clip_val / np.linalg.norm(g))

    ax = axes[1]
    ax.arrow(0, 0, g[0], g[1], head_width=0.15, head_length=0.1, fc='#F44336', ec='#F44336',
             lw=2, label=f'Original g=({g[0]},{g[1]}), ‖g‖={np.linalg.norm(g):.1f}')
    ax.arrow(0, 0, g_val_clip[0], g_val_clip[1], head_width=0.15, head_length=0.1,
             fc='#FF9800', ec='#FF9800', lw=2, label=f'Value-clipped: ({g_val_clip[0]:.1f},{g_val_clip[1]:.1f})')
    ax.arrow(0, 0, g_norm_clip[0], g_norm_clip[1], head_width=0.15, head_length=0.1,
             fc='#4CAF50', ec='#4CAF50', lw=2, label=f'Norm-clipped: preserves direction')
    circle = plt.Circle((0,0), clip_val, fill=False, linestyle='--', color='gray')
    ax.add_patch(circle)
    ax.text(1.5, -0.3, f'clip radius={clip_val}', fontsize=8, color='gray')
    ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 4.5)
    ax.set_aspect('equal'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_title('Value Clip vs Norm Clip\n(norm clip preserves direction)', fontweight='bold')

    plt.suptitle('Gradient Clipping', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('PyTorch usage:')
    print('  nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # norm clipping')
    print('  nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5) # value clipping')
    print('\nRule of thumb: clip_norm=1.0 for RNNs, 5.0 for CNNs, 1.0 for Transformers')

## 8. Early Stopping & Model Checkpointing

### Early Stopping
Monitor validation loss. If it stops improving for `patience` epochs → stop training.

```
  Training loss  ↘ ↘ ↘ ↘ ↘ ↘ ↘ ↘ ↘ ↘  (always decreasing)
  Validation loss ↘ ↘ ↘ ↘ ↗ → ↗ ↗ ↗  (starts rising = overfitting)
                            ↑
                        Best model checkpoint — save here!
```

### Model Checkpointing
Save the model weights whenever validation performance improves.  
After training ends, load the best checkpoint (not the last epoch).

In [ ]:
class EarlyStopping:
    """
    Monitors validation loss and stops training when no improvement
    for `patience` epochs. Saves the best model state_dict.
    """
    def __init__(self, patience=5, min_delta=1e-4, verbose=True):
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_state = None
        self.stop       = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if self.verbose:
                print(f'    ✔ Improved → val_loss={val_loss:.4f}  [checkpoint saved]')
        else:
            self.counter += 1
            if self.verbose:
                print(f'    ✗ No improvement ({self.counter}/{self.patience})')
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f'Restored best model (val_loss={self.best_loss:.4f})')


# Demo with simulated losses
np.random.seed(1)
true_best = 15
train_sim = [2.0 * np.exp(-0.05 * e) + 0.1 * np.random.randn() for e in range(40)]
val_sim   = [2.0 * np.exp(-0.05 * e) + 0.05 * (e / 40)**2 * 3 + 0.1 * np.random.randn() for e in range(40)]
val_sim   = [max(0.1, v) for v in val_sim]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train_sim, 'b-', lw=2, label='Training Loss')
ax.plot(val_sim,   'r-', lw=2, label='Validation Loss')
best_ep = np.argmin(val_sim)
ax.axvline(best_ep, color='green', lw=2, linestyle='--', label=f'Best checkpoint (epoch {best_ep})')
ax.axvspan(best_ep + 5, 40, alpha=0.1, color='red', label='Overfitting zone')
ax.scatter([best_ep], [val_sim[best_ep]], color='green', s=150, zorder=5)
ax.annotate(f'  Best val_loss={val_sim[best_ep]:.3f}\n  early stop here',
            (best_ep, val_sim[best_ep]), xytext=(best_ep+3, val_sim[best_ep]+0.1),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=9, color='green')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Early Stopping: Stop When Validation Loss Stops Improving', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('EarlyStopping usage:')
print('''
  es = EarlyStopping(patience=5)
  for epoch in range(max_epochs):
      train(model)
      val_loss = evaluate(model)
      es.step(val_loss, model)
      if es.stop:
          break
  es.restore_best(model)   # ← always load best checkpoint!
''')

## 9. Full Experiment: Comparing Regularization Strategies on CIFAR-10

We train the same ResNet architecture under 5 different regularization configurations to measure the real impact.

In [ ]:
if TORCH:
    from torchvision.models import resnet18

    # CIFAR-10 loader factory
    def get_cifar_loaders(augment=True, batch_size=128):
        base_tf = [transforms.ToTensor(),
                   transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))]
        if augment:
            train_tf = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, padding=4),
                transforms.ColorJitter(0.2, 0.2, 0.2),
            ] + base_tf)
        else:
            train_tf = transforms.Compose(base_tf)
        test_tf = transforms.Compose(base_tf)
        tr = datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
        te = datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)
        return (DataLoader(tr, batch_size, shuffle=True,  num_workers=0),
                DataLoader(te, 256,        shuffle=False, num_workers=0))

    def build_model():
        """Small ResNet-18 adapted for CIFAR-10 (32×32 input)."""
        m = resnet18(weights=None, num_classes=10)
        m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        return m

    def train_config(name, l2=0.0, use_dropout=False, use_aug=False,
                     use_scheduler=False, epochs=20):
        tr_ld, te_ld = get_cifar_loaders(augment=use_aug)
        model = build_model().to(device)
        if use_dropout:
            model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(512, 10))
        opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=l2)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if use_scheduler else None
        es  = EarlyStopping(patience=5, verbose=False)

        val_accs = []
        for epoch in range(epochs):
            model.train()
            for X, y in tr_ld:
                X, y = X.to(device), y.to(device)
                loss = F.cross_entropy(model(X), y)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                opt.step()
            if sch: sch.step()
            model.eval()
            val_loss = val_correct = 0
            with torch.no_grad():
                for X, y in te_ld:
                    X, y = X.to(device), y.to(device)
                    logits = model(X)
                    val_loss    += F.cross_entropy(logits, y).item() * len(y)
                    val_correct += (logits.argmax(1) == y).sum().item()
            vl = val_loss / len(te_ld.dataset)
            va = val_correct / len(te_ld.dataset) * 100
            val_accs.append(va)
            es.step(vl, model)
            if es.stop: break

        es.restore_best(model)
        final_acc = max(val_accs)
        print(f'{name:<35} best_acc={final_acc:.1f}%  stopped@ep{len(val_accs)}')
        return val_accs

    EPOCHS = 20
    print(f'Training on CIFAR-10 ({EPOCHS} epochs max)\n')
    experiments = [
        ('Baseline (no reg)',           dict(l2=0,    use_dropout=False, use_aug=False,  use_scheduler=False)),
        ('L2=1e-4',                     dict(l2=1e-4, use_dropout=False, use_aug=False,  use_scheduler=False)),
        ('Dropout',                     dict(l2=0,    use_dropout=True,  use_aug=False,  use_scheduler=False)),
        ('Augmentation',                dict(l2=0,    use_dropout=False, use_aug=True,   use_scheduler=False)),
        ('All: L2 + Aug + Sched + ES',  dict(l2=1e-4, use_dropout=False, use_aug=True,   use_scheduler=True)),
    ]

    all_curves = {}
    for name, kw in experiments:
        all_curves[name] = train_config(name, epochs=EPOCHS, **kw)

    # Plot
    palette = ['#9E9E9E','#F44336','#FF9800','#2196F3','#4CAF50']
    fig, ax = plt.subplots(figsize=(12, 5))
    for (name, accs), col in zip(all_curves.items(), palette):
        ax.plot(accs, 'o-', color=col, lw=2, label=f'{name} (best={max(accs):.1f}%)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Regularization Strategies on CIFAR-10 (ResNet-18)', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


Training on CIFAR-10 (20 epochs max) \

Restored best model (val_loss=0.6074) \
&nbsp;Baseline (no reg)                   best_acc=82.6%  stopped@ep11 \
Restored best model (val_loss=0.5532) \
&nbsp;L2=1e-4                             best_acc=83.5%  stopped@ep15 \
Restored best model (val_loss=0.5713) \
&nbsp;Dropout                             best_acc=81.7%  stopped@ep9 \
Restored best model (val_loss=0.3081) \
&nbsp;Augmentation                        best_acc=90.4%  stopped@ep20 \
Restored best model (val_loss=0.2717) \
&nbsp;All: L2 + Aug + Sched + ES          best_acc=92.2%  stopped@ep20 \

## 10. Practical Training Checklist

Use this checklist before starting any serious training run:

### Architecture
- [ ] Batch Normalization after every Conv/Linear layer (before activation)
- [ ] ReLU or GELU activations (avoid sigmoid/tanh in deep layers)
- [ ] Global Average Pooling instead of Flatten+FC for CNNs
- [ ] Skip connections if depth > 10 layers

### Regularization
- [ ] L2 weight decay (`weight_decay=1e-4` in optimizer)
- [ ] Dropout in FC layers (`p=0.5`) — usually NOT in conv layers
- [ ] Data augmentation for vision tasks
- [ ] Early stopping with patience=5–10

### Optimization
- [ ] Adam or AdamW optimizer (not vanilla SGD for most tasks)
- [ ] Cosine Annealing LR schedule
- [ ] Gradient norm clipping (max_norm=1.0 for RNNs, 5.0 for CNNs)
- [ ] Learning rate warmup for first 5–10% of training steps

### Monitoring
- [ ] Log both train AND validation loss/accuracy every epoch
- [ ] Save best model checkpoint (not last epoch)
- [ ] Watch for divergence in first 10 steps — reduce LR if loss increases
- [ ] Check that train_loss decreases in epoch 1 — sanity check

---

## Summary

| Technique | When to use | Typical hyperparameter |
|---|---|---|
| **L2 weight decay** | Always (default) | `1e-4` |
| **L1 regularization** | Sparse weights needed | `1e-5` |
| **Dropout** | FC layers in large networks | `p=0.5` |
| **Batch Normalization** | Always in deep networks | Default parameters |
| **Data augmentation** | Always in vision | Task-specific transforms |
| **Cosine LR schedule** | Most tasks | `T_max=total_epochs` |
| **Gradient clipping** | RNNs, very deep networks | `max_norm=1.0` |
| **Early stopping** | When epochs are uncertain | `patience=5–10` |

---
**Next notebook →** Computer Vision Applications — object detection (YOLO), semantic segmentation (U-Net), and face recognition.